# V82 — soft annotator-mean 目標 refiner: fold2 / arm=soft / seed=8407

GT インスタンスを IoU>0.3 でクラスタ化し、`S_k = (1/R) Σ_r M_{k,r}` の soft 目標で学習。
モデル・損失・後処理は現行 refiner と同一。目的は「soft/クラスタ化された目標が
near-miss(IoU 0.3〜0.5)を TP に押し上げるか」の検証。採用ゲートは
fold2 OOF PQ vs 基準 0.4471、さらに 3fold・両seed符号一致。

GPU必須。


In [ ]:
# [1/7] Drive接続、設定、前提の確認
import json, math, random, shutil, time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from google.colab import drive
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

drive.mount("/content/drive")

class CFG:
    SEED = 8407
    FOLD = 2
    ARM = "soft"
    CLUSTER_IOU = 0.30
    DRIVE_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
    DATA_CACHE = DRIVE_ROOT / "competition_data"
    ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
    PREPROC_CACHE = DRIVE_ROOT / "preprocessed_v30_q96"
    LOCAL_ROOT = Path("/content/solar_v82")
    PREPROC_DIR = LOCAL_ROOT / "preprocessed"
    # --- 以下は build_v35_refiner_folds12.py と同一。触らないこと ---
    CROP_SIZE = 320
    CROP_CONTEXT = 1.65
    REFINER_EPOCHS = 14
    REFINER_LR = 2e-4
    REFINER_BATCH_CANDIDATES = (8, 6, 4, 2)
    REPEAT_FACTOR = 1.5   # record 単位(≈8.2k)と反復数を揃える
    # 確定後処理(README.md「現在地」節)
    CONF = 0.26
    REFINER_THRESHOLD = 0.45
    THRESHOLD_SWEEP = (0.35, 0.4, 0.45, 0.5)
    DROP_THRESHOLD = 0.90
    MIN_AREA = 20
    BASELINE_PQ = 0.4471

CFG.LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
CFG.PREPROC_DIR.mkdir(parents=True, exist_ok=True)
random.seed(CFG.SEED); np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED)

required = {
    "competition data": CFG.DATA_CACHE,
    "grouped_folds.csv": CFG.ARTIFACT_ROOT / "grouped_folds.csv",
    "前処理画像": CFG.PREPROC_CACHE,
    "fold2 標準refiner(比較基準、任意)": CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold2_resume.pt",
    "fold2 YOLO(OOF推論用)": CFG.DRIVE_ROOT / "runs" / f"yolo_fold_2" / "weights" / "best.pt",
}
missing = {k: str(v) for k, v in required.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f"前提artifactが見つかりません: {missing}")
assert torch.cuda.is_available(), "GPUランタイムを選択してください"
print({k: "OK" for k in required})
print({"GPU": torch.cuda.get_device_name(0), "arm": CFG.ARM, "seed": CFG.SEED})


In [ ]:
# [2/7] COCOとfoldを読み、前処理画像をローカルSSDへ展開
def find_dataset_root(base):
    matches = list(Path(base).rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    if not matches:
        raise FileNotFoundError(f"学習annotation JSONが見つかりません: {base}")
    return matches[0].parents[2]

TRAIN_JSON = next(find_dataset_root(CFG.DATA_CACHE).rglob(
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
with open(TRAIN_JSON, encoding="utf-8") as handle:
    coco = json.load(handle)

images = {item["id"]: item for item in coco["images"]}
annotations = defaultdict(list)
for ann in coco["annotations"]:
    annotations[ann["image_id"]].append(ann)
file_to_ids = defaultdict(list)
for image_id, info in images.items():
    file_to_ids[info["file_name"]].append(image_id)
train_files = sorted(file_to_ids)

fold_frame = pd.read_csv(CFG.ARTIFACT_ROOT / "grouped_folds.csv")
fold_of_file = dict(zip(fold_frame["file_name"], fold_frame["fold"].astype(int)))
assert all(name in fold_of_file for name in train_files), "fold未割当の観測がある"

for name in tqdm(train_files, desc="前処理画像をSSDへ"):
    local = CFG.PREPROC_DIR / name
    if not local.exists():
        shutil.copy2(CFG.PREPROC_CACHE / name, local)

try:
    from pycocotools import mask as coco_mask
except ImportError:
    !pip -q install pycocotools
    from pycocotools import mask as coco_mask

def ann_mask(image_id, ann):
    info = images[image_id]
    height, width = int(info["height"]), int(info["width"])
    segmentation = ann["segmentation"]
    if isinstance(segmentation, list):
        rle = coco_mask.merge(coco_mask.frPyObjects(segmentation, height, width))
    elif isinstance(segmentation.get("counts"), list):
        rle = coco_mask.frPyObjects(segmentation, height, width)
    else:
        rle = segmentation
    return coco_mask.decode(rle).astype(np.uint8)

def ann_rle(image_id, ann):
    info = images[image_id]
    height, width = int(info["height"]), int(info["width"])
    segmentation = ann["segmentation"]
    if isinstance(segmentation, list):
        return coco_mask.merge(coco_mask.frPyObjects(segmentation, height, width))
    if isinstance(segmentation.get("counts"), list):
        return coco_mask.frPyObjects(segmentation, height, width)
    rle = dict(segmentation)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("utf-8")
    return rle

print({"観測": len(train_files),
       "fold別": {f: sum(v == f for v in fold_of_file.values()) for f in (0, 1, 2)},
       "annotation": len(coco["annotations"])})


In [ ]:
# [3/7] MediSegUNet (solar_filament/models/mediseg_models.py と同一)
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


In [ ]:
# [3.5/7] soft_target ユーティリティ (solar_filament/pipeline/soft_target.py と同一)
"""Soft annotator-mean targets for v82 refiner training (PLAN §7.2).

One physical filament appears as one instance in *each* annotator record, with
slightly different boundaries. Record-level hard targets teach the refiner to
chase individual annotator noise; the cluster-level soft target

    S_k(x) = (1/R) Σ_r union_i M_{k,r,i}(x)

is the probability that a random annotator marks pixel ``x`` (R = number of
annotator records for the image; records that did not draw the cluster count
zero). Predicting the distribution mean recovers the 0.5-IoU cliff on
near-miss candidates — the mechanism the near-miss analysis points at.

Clustering unions instances whose pairwise IoU exceeds ``iou_threshold``,
but never merges two instances from the *same* record: if one annotator drew
two overlapping masks they considered them distinct filaments.

Full-frame soft masks are expensive (float32, 16 MB each at 2048²), so the
training dataset should call :func:`soft_target_crop` with the cluster's crop
box and build the target only inside it.
"""


from typing import Sequence

import numpy as np

DEFAULT_CLUSTER_IOU = 0.30


def _iou(a: np.ndarray, b: np.ndarray) -> float:
    inter = int(np.logical_and(a, b).sum())
    if inter == 0:
        return 0.0
    return float(inter) / float(a.sum() + b.sum() - inter)


def cluster_gt_instances(
    masks_per_record: Sequence[Sequence[np.ndarray]],
    iou_threshold: float = DEFAULT_CLUSTER_IOU,
) -> list[list[tuple[int, int]]]:
    """Union-find clusters of GT instances across annotator records.

    ``masks_per_record[r]`` is the instance list of record ``r``. Returns a
    list of clusters; each cluster is a list of ``(record, instance)`` pairs.
    Instances sharing a record are never directly merged.
    """
    parent: dict[tuple[int, int], tuple[int, int]] = {}

    def find(x: tuple[int, int]) -> tuple[int, int]:
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            parent[x], x = root, parent[x]
        return root

    instances = [
        (r, i)
        for r, masks in enumerate(masks_per_record)
        for i in range(len(masks))
    ]
    for inst in instances:
        parent[inst] = inst
    for pos, (r, i) in enumerate(instances):
        for s, j in instances[pos + 1 :]:
            if s == r:
                continue
            if _iou(masks_per_record[r][i], masks_per_record[s][j]) > iou_threshold:
                ra, rb = find((r, i)), find((s, j))
                if ra != rb:
                    parent[rb] = ra
    clusters: dict[tuple[int, int], list[tuple[int, int]]] = {}
    for inst in instances:
        clusters.setdefault(find(inst), []).append(inst)
    return list(clusters.values())


def cluster_bbox(
    masks_per_record: Sequence[Sequence[np.ndarray]],
    cluster: Sequence[tuple[int, int]],
) -> tuple[int, int, int, int]:
    """Union bounding box ``(x0, y0, x1, y1)`` of a cluster's instances."""
    x0 = y0 = np.iinfo(np.int64).max
    x1 = y1 = 0
    for r, i in cluster:
        ys, xs = np.nonzero(masks_per_record[r][i])
        x0 = min(x0, int(xs.min()))
        x1 = max(x1, int(xs.max()) + 1)
        y0 = min(y0, int(ys.min()))
        y1 = max(y1, int(ys.max()) + 1)
    return x0, y0, x1, y1


def soft_target_crop(
    masks_per_record: Sequence[Sequence[np.ndarray]],
    cluster: Sequence[tuple[int, int]],
    box: tuple[int, int, int, int],
) -> np.ndarray:
    """Soft target ``S_k`` restricted to ``box`` = (x0, y0, x1, y1), float32.

    A record contributing several instances to the cluster contributes their
    union once (one annotator, one opinion).
    """
    x0, y0, x1, y1 = box
    acc = np.zeros((y1 - y0, x1 - x0), dtype=np.float32)
    by_record: dict[int, list[int]] = {}
    for r, i in cluster:
        by_record.setdefault(r, []).append(i)
    for r, indices in by_record.items():
        union = np.logical_or.reduce(
            [masks_per_record[r][i][y0:y1, x0:x1] for i in indices]
        )
        acc += union.astype(np.float32)
    return acc / max(len(masks_per_record), 1)


def build_soft_targets(
    masks_per_record: Sequence[Sequence[np.ndarray]],
    iou_threshold: float = DEFAULT_CLUSTER_IOU,
) -> tuple[list[np.ndarray], list[list[tuple[int, int]]]]:
    """Full-frame soft targets + clusters. For tests and small canvases; the
    training path should prefer :func:`soft_target_crop` to avoid holding many
    float32 canvases."""
    clusters = cluster_gt_instances(masks_per_record, iou_threshold)
    if not clusters:
        return [], []
    height, width = masks_per_record[0][0].shape
    soft = []
    for cluster in clusters:
        target = np.zeros((height, width), dtype=np.float32)
        box = cluster_bbox(masks_per_record, cluster)
        x0, y0, x1, y1 = box
        target[y0:y1, x0:x1] = soft_target_crop(masks_per_record, cluster, box)
        soft.append(target)
    return soft, clusters


In [ ]:
# [3.7/7] 形状サニティチェック(学習開始前に必ず確認する)
_probe = MediSegUNet(3, 1, False).cuda()
_dummy = torch.randn(2, 3, CFG.CROP_SIZE, CFG.CROP_SIZE).cuda()
with torch.no_grad():
    _out = _probe(_dummy)
print("output shape:", _out.shape, "(期待値: [2, 1, 320, 320])")
assert _out.shape == (2, 1, CFG.CROP_SIZE, CFG.CROP_SIZE), "出力形状が想定と異なる"
del _probe, _dummy, _out
torch.cuda.empty_cache()
print("形状チェックOK")


In [ ]:
# [4/7] ClusterCropDataset・loss・学習ループ
def square_box_from_bbox(bbox, height, width, context=CFG.CROP_CONTEXT, minimum=96):
    x, y, w, h = map(float, bbox)
    cx, cy = x + w / 2, y + h / 2
    side = int(math.ceil(max(w, h) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

class ClusterCropDataset(Dataset):
    """(file, cluster) 単位の soft/union 目標。record 単位の InstanceCropDataset
    と同じ画像側処理で、目標だけが変わる。検証セットも同じコードを通る
    (augment=False で完全に決定的)"""

    def __init__(self, file_names, augment=False, repeat_factor=CFG.REPEAT_FACTOR):
        self.augment = augment
        self.files = {}
        self.samples = []
        for file_name in tqdm(file_names, desc="GTクラスタ化"):
            ids = file_to_ids[file_name]
            rles = [[ann_rle(i, a) for a in annotations[i] if not a.get("iscrowd", 0)]
                    for i in ids]
            # クラスタ化だけのために decode し、RLE のみ保持する (2048² bool を
            # 全件保持すると ~35GB になるため)
            masks = [[coco_mask.decode(r).astype(bool) for r in rr] for rr in rles]
            clusters = cluster_gt_instances(masks, CFG.CLUSTER_IOU)
            self.files[file_name] = (rles, clusters, len(ids))
            for cluster_index in range(len(clusters)):
                self.samples.append((file_name, cluster_index))
        target_len = int(len(self.samples) * repeat_factor)
        base = list(self.samples)
        while len(self.samples) < target_len:
            need = target_len - len(self.samples)
            self.samples += base[:need]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        file_name, cluster_index = self.samples[index]
        rles, clusters, n_records = self.files[file_name]
        cluster = clusters[cluster_index]
        info_id = file_to_ids[file_name][0]
        height, width = int(images[info_id]["height"]), int(images[info_id]["width"])

        # クラスタの union bbox → square box。decode はクラスタメンバだけに
        # 絞る (モジュール関数は cluster の (r,i) しか参照しないため None でよい)。
        member_idx = defaultdict(set)
        for r, i in cluster:
            member_idx[r].add(i)
        # member_idx[r] (subscript): defaultdict が空 set を返す。.get(r) だと
        # 未登録 record で None になり `i in None` で TypeError するので注意。
        masks_per_record = [
            [coco_mask.decode(rr[i]).astype(bool) if i in member_idx[r] else None
             for i in range(len(rr))]
            for r, rr in enumerate(rles)
        ]
        bx0, by0, bx1, by1 = cluster_bbox(masks_per_record, cluster)
        bbox = [float(bx0), float(by0), float(bx1 - bx0), float(by1 - by0)]
        x0, y0, x1, y1 = square_box_from_bbox(bbox, height, width)

        image = cv2.imread(str(CFG.PREPROC_DIR / file_name), cv2.IMREAD_COLOR)
        image = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE),
                           interpolation=cv2.INTER_LINEAR)
        image = torch.from_numpy(image.transpose(2, 0, 1).copy()).float() / 255.0

        soft_np = soft_target_crop(masks_per_record, cluster, (x0, y0, x1, y1))
        if CFG.ARM == "soft":
            target_np = cv2.resize(soft_np, (CFG.CROP_SIZE, CFG.CROP_SIZE),
                                   interpolation=cv2.INTER_LINEAR)
        else:  # hard_union: 1recordでも塗れば1 (S>0 と同値)
            target_np = cv2.resize((soft_np > 0).astype(np.uint8),
                                   (CFG.CROP_SIZE, CFG.CROP_SIZE),
                                   interpolation=cv2.INTER_NEAREST).astype(np.float32)
        target = torch.from_numpy(target_np[None].copy()).float()

        if self.augment:
            k = random.randrange(4)
            image, target = torch.rot90(image, k, (1, 2)), torch.rot90(target, k, (1, 2))
            if random.random() < 0.5:
                image, target = torch.flip(image, (2,)), torch.flip(target, (2,))
            if random.random() < 0.5:
                image, target = torch.flip(image, (1,)), torch.flip(target, (1,))
            if random.random() < 0.5:
                image[:2] = torch.clamp(image[:2] ** random.uniform(0.85, 1.20), 0, 1)
        return image, target

def soft_dice_loss(logits, targets):
    probs = logits.sigmoid()
    intersection = (probs * targets).sum((1, 2, 3))
    denominator = probs.sum((1, 2, 3)) + targets.sum((1, 2, 3))
    return (1 - (2 * intersection + 1) / (denominator + 1)).mean()

def boundary_map(tensor):
    dilation = F.max_pool2d(tensor, 5, stride=1, padding=2)
    erosion = -F.max_pool2d(-tensor, 5, stride=1, padding=2)
    return torch.clamp(dilation - erosion, 0, 1)

def refiner_loss(logits, targets):
    region = 0.35 * F.binary_cross_entropy_with_logits(logits, targets) + 0.65 * soft_dice_loss(logits, targets)
    edge = F.l1_loss(boundary_map(logits.sigmoid()), boundary_map(targets))
    return region + 0.15 * edge

def crop_dice(model, loader, threshold=0.5):
    model.eval()
    total = count = 0
    with torch.no_grad():
        for image, target in loader:
            image, target = image.cuda(non_blocking=True), target.cuda(non_blocking=True).bool()
            pred = model(image).sigmoid() >= threshold
            intersection = (pred & target).sum((1, 2, 3)).float()
            denominator = pred.sum((1, 2, 3)) + target.sum((1, 2, 3))
            total += ((2 * intersection + 1) / (denominator + 1)).sum().item()
            count += len(image)
    return total / max(count, 1)

def choose_batch_size(dataset):
    for candidate in CFG.REFINER_BATCH_CANDIDATES:
        try:
            loader = DataLoader(dataset, batch_size=candidate, shuffle=True, num_workers=2, pin_memory=True)
            image, target = next(iter(loader))
            probe = MediSegUNet(3, 1, False).cuda()
            with torch.amp.autocast("cuda"):
                loss = refiner_loss(probe(image.cuda()), target.cuda())
            loss.backward()
            del probe, image, target, loss, loader
            torch.cuda.empty_cache()
            return candidate
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
    raise RuntimeError("どのbatch sizeもGPUメモリに収まらない")


In [ ]:
# [5/7] fold2を学習(該当foldを検証専用に除外)
started = time.time()
ckpt_name = f"mediseg_soft_refiner_fold2_s8407_resume.pt"
checkpoint_path = CFG.ARTIFACT_ROOT / ckpt_name
train_files_fold = [n for n in train_files if fold_of_file[n] != CFG.FOLD]
valid_files_fold = [n for n in train_files if fold_of_file[n] == CFG.FOLD]
train_ds = ClusterCropDataset(train_files_fold, augment=True)
valid_ds = ClusterCropDataset(valid_files_fold, augment=False, repeat_factor=1.0)
assert not (set(train_files_fold) & set(valid_files_fold)), "fold分割が壊れている"
print({"fold": CFG.FOLD, "arm": CFG.ARM, "seed": CFG.SEED,
       "train_crops": len(train_ds), "valid_crops": len(valid_ds)})

batch_size = choose_batch_size(train_ds)
print("batch_size =", batch_size)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)

model = MediSegUNet(3, 1, False).cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.REFINER_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.REFINER_EPOCHS)
scaler = torch.amp.GradScaler("cuda")
best_dice, best_state = -1.0, None

for epoch in range(CFG.REFINER_EPOCHS):
    model.train()
    running = 0.0
    progress = tqdm(train_loader, desc=f"fold{CFG.FOLD} {CFG.ARM} refiner {epoch + 1}/{CFG.REFINER_EPOCHS}")
    for image, target in progress:
        image, target = image.cuda(non_blocking=True), target.cuda(non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            loss = refiner_loss(model(image), target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += loss.item()
        progress.set_postfix(loss=f"{running / (progress.n + 1):.4f}")
    scheduler.step()
    dice = crop_dice(model, valid_loader)
    if dice > best_dice:
        best_dice = dice
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    torch.save({"epoch": epoch, "model": model.state_dict(), "best_model": best_state,
                "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "best_dice": best_dice, "arm": CFG.ARM, "seed": CFG.SEED}, checkpoint_path)
    print(f"fold{CFG.FOLD} {CFG.ARM} s{CFG.SEED} epoch {epoch + 1}: crop Dice {dice:.6f} (best {best_dice:.6f})")

best_crop_dice = best_dice
print({"checkpoint": str(checkpoint_path), "best_crop_dice": best_crop_dice,
       "minutes": round((time.time() - started) / 60, 1)})


In [ ]:
# [6/7] fold2 OOF(リークなし)を確定後処理で採点し、基準0.4471と比較
# refiner 閾値は 0.45 が主結果、{0.35,0.40,0.45,0.50} は事前宣言の補助掃引 (§7.2.4)
!pip -q install ultralytics
from ultralytics import YOLO

yolo_model = YOLO(str(CFG.DRIVE_ROOT / "runs" / f"yolo_fold_2" / "weights" / "best.pt"))

refiner = MediSegUNet(3, 1, False).cuda().eval()
ckpt = torch.load(checkpoint_path, map_location="cuda", weights_only=False)
refiner.load_state_dict(ckpt.get("best_model") or ckpt["model"], strict=True)

def result_masks(result):
    if result.masks is None:
        return [], np.zeros(0, np.float32)
    height, width = result.orig_shape
    output = []
    for mask in result.masks.data.detach().cpu().numpy():
        if mask.shape != (height, width):
            mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_LINEAR)
        output.append(mask >= 0.5)
    confidences = result.boxes.conf.detach().cpu().numpy().astype(np.float32) if result.boxes is not None else np.zeros(0, np.float32)
    return output, confidences

oof_files = [name for name in train_files if fold_of_file[name] == CFG.FOLD]

def mask_square_box(mask, context=1.65, minimum=96):
    height, width = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, width, height
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

@torch.no_grad()
def refine_probability(image, proposal):
    x0, y0, x1, y1 = mask_square_box(proposal)
    crop = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
    tensor = torch.from_numpy(crop.transpose(2, 0, 1)[None].copy()).float().cuda() / 255.0
    probability = refiner(tensor).sigmoid()[0, 0].cpu().numpy()
    return cv2.resize(probability, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR), (x0, y0, x1, y1)

def greedy_disjoint(masks, order, min_area=CFG.MIN_AREA):
    occupied = np.zeros((2048, 2048), dtype=bool)
    kept = []
    for index in order:
        exclusive = masks[index] & ~occupied
        if exclusive.sum() < min_area:
            continue
        kept.append(exclusive)
        occupied |= exclusive
    return kept

from pycocotools import mask as mask_utils

def drop_contained_indices(masks, threshold=CFG.DROP_THRESHOLD):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), dtype=np.float64)
    self_iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped:
                continue
            value = self_iou[i, j]
            if value <= 0:
                continue
            intersection = value * (areas[i] + areas[j]) / (1.0 + value)
            smaller = i if areas[i] <= areas[j] else j
            if intersection / max(areas[smaller], 1.0) >= threshold:
                dropped.add(smaller)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership(masks, scores):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], dtype=np.float64)
    subset = drop_contained_indices(masks)
    return greedy_disjoint(masks, sorted(subset, key=lambda i: areas[i]))

def to_rle(annotation, height, width):
    segmentation = annotation["segmentation"]
    if isinstance(segmentation, list):
        return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))
    if isinstance(segmentation.get("counts"), list):
        return mask_utils.frPyObjects(segmentation, height, width)
    rle = dict(segmentation)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("utf-8")
    return rle

def iou_matrix(truth, pred):
    if not truth or not pred:
        return np.zeros((len(truth), len(pred)), np.float64)
    return np.asarray(mask_utils.iou(pred, truth, [0] * len(truth)), dtype=np.float64).T

class PanopticQuality:
    def __init__(self, iou_threshold=0.5):
        self.iou_threshold = iou_threshold
        self.tp_ious = []
        self.fp_count = 0
        self.fn_count = 0
    def add_matrix(self, matrix, n_gt, n_pred):
        if n_gt == 0:
            self.fp_count += n_pred; return
        if n_pred == 0:
            self.fn_count += n_gt; return
        hits = matrix > self.iou_threshold
        scores = matrix[hits].tolist()
        self.tp_ious.extend(scores)
        self.fp_count += int((hits.sum(axis=0) == 0).sum())
        self.fn_count += int((hits.sum(axis=1) == 0).sum())
    @property
    def denominator(self):
        return len(self.tp_ious) + 0.5 * self.fp_count + 0.5 * self.fn_count
    @property
    def score(self):
        d = self.denominator
        return float(sum(self.tp_ious) / d) if d > 0 else 0.0
    def summary(self):
        d = self.denominator
        return {"pq": self.score, "tp": len(self.tp_ious), "fp": self.fp_count, "fn": self.fn_count, "denominator": d}

gt_rles = {}
for image_id, info in images.items():
    height, width = int(info["height"]), int(info["width"])
    gt_rles[image_id] = [to_rle(ann, height, width) for ann in annotations[image_id] if not ann.get("iscrowd", 0)]

# refiner の確率マップは閾値ごとに再利用するため先に全件計算して保持する
cache = {}
for name in tqdm(oof_files, desc="OOF推論(確率マップを保持)"):
    image_path = CFG.PREPROC_DIR / name
    result = yolo_model.predict(str(image_path), imgsz=1280, conf=0.08, iou=0.55,
                                 agnostic_nms=True, retina_masks=True, verbose=False)[0]
    masks, confidences = result_masks(result)
    keep = [i for i, (m, s) in enumerate(zip(masks, confidences)) if s >= CFG.CONF and m.sum() >= CFG.MIN_AREA]
    proposals = [masks[i] for i in keep]
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    cache[name] = [(refine_probability(image, m), m) for m in proposals]

meters = {t: PanopticQuality() for t in CFG.THRESHOLD_SWEEP}
for name in oof_files:
    for threshold in CFG.THRESHOLD_SWEEP:
        refined = []
        for (probability, box), proposal in cache[name]:
            x0, y0, x1, y1 = box
            mask = np.zeros_like(proposal, dtype=bool)
            mask[y0:y1, x0:x1] = probability >= threshold
            refined.append(mask & proposal)  # shrink
        predictions = apply_ownership(refined, np.ones(len(refined)))
        pred_rles = []
        if predictions:
            pred_rles = mask_utils.encode(np.asfortranarray(np.stack(predictions, axis=-1).astype(np.uint8)))
        for image_id in file_to_ids[name]:
            truth = gt_rles[image_id]
            matrix = iou_matrix(truth, pred_rles)
            meters[threshold].add_matrix(matrix, len(truth), len(pred_rles))

report = {}
for threshold in CFG.THRESHOLD_SWEEP:
    summary = meters[threshold].summary()
    report[str(threshold)] = {"pq": round(summary["pq"], 6),
                              "delta_vs_baseline": round(summary["pq"] - CFG.BASELINE_PQ, 6)}
print(json.dumps({"fold": CFG.FOLD, "arm": CFG.ARM, "seed": CFG.SEED,
                  "baseline_pq": CFG.BASELINE_PQ, "by_threshold": report,
                  "best_crop_dice": best_crop_dice}, ensure_ascii=False, indent=2))


In [ ]:
# [7/7] manifest保存
import hashlib

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    "experiment_id": "v82_soft_target_refiner",
    "arm": CFG.ARM, "fold": CFG.FOLD, "seed": CFG.SEED,
    "cluster_iou": CFG.CLUSTER_IOU, "repeat_factor": CFG.REPEAT_FACTOR,
    "model": "MediSegUNet(3,1) — 現行 refiner と同一",
    "epochs": CFG.REFINER_EPOCHS, "lr": CFG.REFINER_LR,
    "best_crop_dice": best_crop_dice,
    "baseline_pq": CFG.BASELINE_PQ,
    "oof_pq_by_threshold": report,
    "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
    "checkpoint": str(checkpoint_path), "checkpoint_sha256": sha256_of(checkpoint_path),
    "gate": "soft - hard_record が 3fold・両seedで正、pooled CI がゼロを除外",
}
report_path = CFG.ARTIFACT_ROOT / f"v82_{CFG.ARM}_fold{CFG.FOLD}_s{CFG.SEED}_manifest.json"
report_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print(f"\n保存: {report_path}")
